# Critical Input DEQN: Compare Policy Environments

This notebook loads saved outputs from the fixed Taylor, bottleneck-adjusted Taylor, discretion, and commitment notebooks and writes a compact comparison table.

In [ ]:
# Locate saved diagnostics for all trained policy environments.
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
COMPARISON_DIR = ARTIFACT_ROOT / 'comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    'fixed_taylor': ARTIFACT_ROOT / 'fixed_taylor' / 'fixed_eval.json',
    'modified_taylor': ARTIFACT_ROOT / 'modified_taylor' / 'ba_eval.json',
    'discretion': ARTIFACT_ROOT / 'discretion' / 'discretion_eval.json',
    'commitment': ARTIFACT_ROOT / 'commitment' / 'commitment_eval.json',
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing {name} results: {path}')

In [ ]:
# Collect key residual diagnostics into one comparison table.
rows = []
for policy, path in paths.items():
    with path.open('r', encoding='utf-8') as fh:
        d = json.load(fh)
    row = {'policy': policy}
    for key in [
        'overall.rms', 'overall.max_abs', 'hh_euler.rms',
        'resource.rms', 'price_index.rms', 'calvo_S.rms', 'calvo_F.rms',
        'Q.rms', 'exact_cap_product_scaled.rms',
        'exact_repair_projection.rms', 'priv_Q.rms',
        'exact_cap_gap_negative.max_abs', 'exact_repair_projection.max_abs',
        'bellman.rms', 'stat_R.rms', 'stat_Pi.rms', 'stat_Q_A.rms',
        'promise_E.rms', 'promise_S.rms', 'promise_F.rms', 'promise_Q.rms',
    ]:
        row[key] = d.get(key)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('policy')
summary

In [ ]:
# Save the comparison table for later inspection.
out_csv = COMPARISON_DIR / 'rule_eval_summary.csv'
summary.to_csv(out_csv)
out_csv